In [0]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F

@dp.materialized_view(
    name="users_cleaned",
    comment="Cleaned users data with fixed dates and removed duplicates"
)
def users_cleaned():
    # load the data from the table
    df = spark.table('data_engineering.video1.users_dirty_csv')
    
    # Fix the date column: replace '.' with '/' separator
    df = df.withColumn(
        "signup_date",
        F.regexp_replace(F.col("signup_date"), "\\.", "/")
    )
    
    # Convert to date format (handles MM/dd/yy format)
    df = df.withColumn(
        "signup_date",
        F.to_date(F.col("signup_date"), "MM/dd/yy")
    )
    
    # Remove duplicates based on user_id, keeping the first occurrence
    df = df.dropDuplicates(["user_id"])
    
    return df